In [10]:
import jax 
import jax.numpy as jnp
from functools import partial

In [5]:
ctx_len = 10
one = jnp.arange(0, -2 * ctx_len - 1, -1)

In [6]:
one + one

Array([  0,  -2,  -4,  -6,  -8, -10, -12, -14, -16, -18, -20, -22, -24,
       -26, -28, -30, -32, -34, -36, -38, -40], dtype=int32)

In [7]:
jnp.concatenate([one, one], axis=0)

Array([  0,  -1,  -2,  -3,  -4,  -5,  -6,  -7,  -8,  -9, -10, -11, -12,
       -13, -14, -15, -16, -17, -18, -19, -20,   0,  -1,  -2,  -3,  -4,
        -5,  -6,  -7,  -8,  -9, -10, -11, -12, -13, -14, -15, -16, -17,
       -18, -19, -20], dtype=int32)

In [8]:
one.shape

(21,)

In [38]:
class TestClass:

    def __init__(self, ctx_len, n_topics):
        self.ctx_len = ctx_len
        self.n_topics = n_topics
        
    @partial(jax.jit, static_argnums=0)
    def _get_context_tensor(self, *, batch: jax.Array) -> jax.Array:
        """
        Stacks 2d-data into a 3d-tensor along a new (context) axis,
        shifting the data along the new axis. The constructed tensor
        if helpful for fast context convolution with given weights.
        """
        batch_size = batch.shape[0]     # размерность батча - количество токенов
        pad_token = -1  # assuming we don't have negative tokens in vocabulary

        # shifts for rolling the batch along new dimension
        shifts = jnp.arange(0, -2 * self.ctx_len - 1, -1)  # (2C + 1, ) [0, -1, -2, ..., -2 * self.ctx_len - 1]

        # pad batch for shifting
        max_shift = self.ctx_len * 2 + batch_size
        padded_batch = jnp.full(
            (max_shift, self.n_topics),
            fill_value=pad_token,
            dtype=batch.dtype,
        )  # (I + 2C, T)
        padded_batch = padded_batch.at[self.ctx_len:self.ctx_len + batch_size].set(batch)
        print(f"{padded_batch.shape}=")
        jax.debug.print("padded_batch {padded_batch}", padded_batch=padded_batch) 
        '''
        padding = jnp.full(
            (self.ctx_len, self.n_topics),
            fill_value=pad_token,
            dtype=batch.dtype,
        )  # (C, T)
        br_batch = jnp.tile(batch, (self.n_topics, 1))
        print(f"{padding.shape}=")
        print(f"{br_batch.shape}=")
        padded_batch = jnp.concatenate([padding, br_batch, padding], axis=0)
       '''
 
        # rolling and clipping each "slice" of batch
        def shift_batch(shift):
            return jnp.roll(padded_batch, shift, axis=0)[:batch_size]

        # apply vmap over all shifts
        stacked_tensor = jax.vmap(shift_batch)(shifts).transpose(1, 0, 2)
        return stacked_tensor  # (I, 2C + 1, T)


In [44]:
batch = jnp.arange(24).reshape((6, 4))
print(batch)
tc = TestClass(3, 4)
tc._get_context_tensor(batch=batch)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]
 [16 17 18 19]
 [20 21 22 23]]


Array([[[-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]],

       [[-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]],

       [[-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]],

       [[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23],
        [-1, -1, -1, -1]],

       [[ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1]],

       [[ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
  

In [42]:
class TestClass:

    def __init__(self, ctx_len, n_topics):
        self.ctx_len = ctx_len
        self.n_topics = n_topics
        
    @partial(jax.jit, static_argnums=0)
    def _get_context_tensor(self, *, batch: jax.Array) -> jax.Array:
        """
        Stacks 2d-data into a 3d-tensor along a new (context) axis,
        shifting the data along the new axis. The constructed tensor
        if helpful for fast context convolution with given weights.
        """
        batch_size = batch.shape[0]     # размерность батча - количество токенов
        pad_token = -1  # assuming we don't have negative tokens in vocabulary

        # shifts for rolling the batch along new dimension
        shifts = jnp.arange(0, -2 * self.ctx_len - 1, -1)  # (2C + 1, ) [0, -1, -2, ..., -2 * self.ctx_len - 1]

        '''
        # pad batch for shifting
        max_shift = self.ctx_len * 2 + batch_size
        padded_batch = jnp.full(
            (max_shift, self.n_topics),
            fill_value=pad_token,
            dtype=batch.dtype,
        )  # (I + 2C, T)
        padded_batch = padded_batch.at[self.ctx_len:self.ctx_len + batch_size].set(batch)
        print(f"{padded_batch.shape}=")
        '''
        padding = jnp.full(
            (self.ctx_len, self.n_topics),
            fill_value=pad_token,
            dtype=batch.dtype,
        )  # (C, T)

        padded_batch = jnp.concatenate([padding, batch, padding], axis=0)
 
        # rolling and clipping each "slice" of batch
        def shift_batch(shift):
            return jnp.roll(padded_batch, shift, axis=0)[:batch_size]

        # apply vmap over all shifts
        stacked_tensor = jax.vmap(shift_batch)(shifts).transpose(1, 0, 2)
        return stacked_tensor  # (I, 2C + 1, T)


In [43]:
batch = jnp.arange(20).reshape((5, 4))
print(batch)
tc = TestClass(3, 4)
tc._get_context_tensor(batch=batch)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]
 [16 17 18 19]]


Array([[[-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]],

       [[-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]],

       [[-1, -1, -1, -1],
        [ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [-1, -1, -1, -1]],

       [[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1]],

       [[ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1],
        [-1, -1, -1, -1]]], dtype=int32)